# Step 0: Calibrate the simulation

The first version of the simulation produced unrealistic results: CTR rose 14%, funded loans per user rose 44%, and five partners received nearly all applications. Two causes: every user saw nearly the same partner order regardless of credit, and the ranking favored high-approval offers, so showing odds mostly increased clicks.

The revised simulation ranks offers by expected revenue, makes partner strictness depend on credit tier, and models an asymmetric reaction to odds (low odds deter more than high odds encourage). This notebook computes the true expected effects, without sampling noise, for a grid of reaction settings and keeps the one that produces a realistic pattern: CTR down 5 to 15%, funded loans per user up 3 to 8%.

The chosen values (pivot 0.40, up 1.0, down 2.0) are set in `src/simulate.py`.

In [ ]:
# Imports and paths.
import sys
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import src.simulate as sim

In [ ]:
# Build the marketplace and the ranked offers each user sees, using the revised ranking.
users, partners, rng = sim.build_marketplace()
tiers = users["tier"].to_numpy()
mult = partners["approval_mult"].to_numpy()
appeal = partners["appeal"].to_numpy()
payout = partners["payout"].to_numpy()

approval = sim.approval_matrix(tiers, mult)
score = approval * payout[None, :] * np.exp(appeal)[None, :] * rng.lognormal(0, sim.RANK_NOISE, approval.shape)
shown = np.argsort(-score, axis=1)[:, :sim.K]
s_appr = np.take_along_axis(approval, shown, axis=1)
s_appeal, s_pay = appeal[shown], payout[shown]
base = (sim.CLICK_INTERCEPT - sim.POSITION_DECAY * np.arange(sim.K)[None, :]
        + s_appeal + users["intent"].to_numpy()[:, None])
p0 = 1 / (1 + np.exp(-base))

In [ ]:
# True expected lifts for each combination of odds-reaction settings.
# The application rate cancels out of every lift, so it is left out here.
rows = []
for pivot, up, down in product([0.3, 0.4, 0.5], [0.3, 0.6, 1.0], [1.0, 2.0, 3.0]):
    shift = up * np.clip(s_appr - pivot, 0, None) - down * np.clip(pivot - s_appr, 0, None)
    p1 = 1 / (1 + np.exp(-(base + shift)))
    c0, c1 = p0.sum(1).mean(), p1.sum(1).mean()
    f0, f1 = (p0 * s_appr).sum(1).mean(), (p1 * s_appr).sum(1).mean()
    r0, r1 = (p0 * s_appr * s_pay).sum(1).mean(), (p1 * s_appr * s_pay).sum(1).mean()
    rows.append({"pivot": pivot, "up": up, "down": down,
                 "ctr_lift": c1 / c0 - 1, "funded_lift": f1 / f0 - 1,
                 "revenue_lift": r1 / r0 - 1, "ctf_lift": (f1 / c1) / (f0 / c0) - 1})

grid = pd.DataFrame(rows)
in_range = grid[grid["ctr_lift"].between(-0.15, -0.05) & grid["funded_lift"].between(0.03, 0.08)]
print(f"Settings in the target range: {len(in_range)} of {len(grid)}")
print(in_range.round(3).to_string(index=False))

In [ ]:
# Check that the ranking now mixes partners and differs by credit tier.
# Strict, high-payout partners should appear mainly for excellent and good credit.
slot_share = pd.crosstab(np.repeat(users["tier_name"].to_numpy(), sim.K),
                         partners["partner_id"].to_numpy()[shown.ravel()], normalize="index")
print(slot_share.loc[["excellent", "good", "fair", "poor"]].round(2))